# Day3 · 블록2 · Python 멀티스레딩 · 멀티프로세스 실습

> **강의자료**: `강의자료/03.02.Python-Thread-Process.md`

| Part | 주제 |
|------|------|
| Part 1 | 멀티태스킹 기초 개념 |
| Part 2 | 멀티스레딩 (threading 모듈) |
| Part 3 | 멀티프로세싱 (multiprocessing 모듈) |
| Part 4 | 퀴즈 · 복습 문제 |

이 실습에서는 로봇 시스템에서 여러 작업을 동시에 처리하기 위한
멀티스레딩과 멀티프로세싱의 개념과 사용법을 익힙니다.
센서 읽기, 데이터 공유, Lock·Queue·Event를 활용한 안전한 멀티태스킹 패턴을 실습합니다.

---
## Part 1: 멀티태스킹 기초 개념

로봇이 여러 작업을 동시에 처리해야 하는 이유와
프로세스·스레드·GIL의 핵심 개념을 이해합니다.

### 1.1 왜 멀티태스킹이 필요한가?

로봇은 여러 작업을 동시에 처리해야 하는 복잡한 시스템입니다.

- **동시 센서 읽기 시나리오**
    - 이동 중 초음파 센서로 거리 측정
    - 카메라로 장애물 인식
    - 동시에 바퀴 모터에 명령 전송

- **문제점: Single Thread 방식**
    - 카메라 영상 처리 중(약 100~200ms) 초음파 센서 값을 읽지 못함
    - 이 짧은 지연 시간 동안 로봇이 장애물을 발견하지 못하고 충돌 가능

- **해결책: 멀티태스킹**
    - **I/O 바운드 작업** 처리 시 응답을 기다리는 동안 다른 작업 수행
    - 시스템의 **응답성** 극대화

### 1.2 프로세스(Process) vs 스레드(Thread)

"주방의 요리사"로 이해하기

- **프로세스 (Process)**
    - 실행 중인 프로그램의 **독립적인 인스턴스**
    - **독립된 메모리 공간**과 Python 인터프리터 보유
    - 비유: 별도의 주방과 도구를 가진 요리사
    - 한 요리사가 도구를 망가뜨려도 다른 주방에 영향 없음
    - 데이터 전달 시 `Queue` 같은 채널 필요, 생성 자원 소모 큼

- **스레드 (Thread)**
    - 하나의 프로세스 내에서 실행되는 **여러 흐름**
    - 같은 프로세스 내의 **메모리 공간 공유**
    - 비유: 한 주방에서 같은 조리 도구·재료를 공유하며 작업하는 여러 요리사
    - 데이터 공유가 쉬운 반면, 동시 접근 시 문제 발생 가능 → 관리 필요

### 1.3 Python GIL (Global Interpreter Lock)

**GIL: Python의 독특한 특징**

- **GIL이란?**
    - 한 번에 하나의 네이티브 스레드만 Python 바이트코드를 실행하도록 제한하는 '잠금장치'

- **CPU 집약적 작업 (이미지 처리, 경로 계획 등)**
    - 멀티스레딩으로 돌려도 실제 속도가 빨라지지 않음
    - → `multiprocessing` 모듈 사용 필요

- **I/O 바운드 작업 (센서 통신, 네트워크)**
    - CPU를 거의 쓰지 않고 대기 시간이 많음
    - GIL은 대기 시간에 다른 스레드에게 실행권을 넘김
    - → **로봇의 여러 센서를 동시에 관리하는 데 멀티스레딩이 매우 효율적**

### 1.4 멀티스레딩 vs 멀티프로세싱 선택 기준

| 구분 | 멀티스레딩 | 멀티프로세싱 |
|------|-----------|------------|
| **메모리** | 프로세스 내 공유 | 각 프로세스 독립 |
| **GIL 영향** | 받음 | 받지 않음 |
| **적합한 작업** | I/O 바운드 | CPU 바운드 |
| **생성 비용** | 낮음 | 높음 |
| **데이터 공유** | 쉬움 (Lock 필요) | Queue/Pipe 필요 |
| **사용 예시** | 시리얼 포트 읽기, 네트워크 요청, 하트비트 | 이미지 처리, AI 추론, 경로 계획 |

---
## Part 2: 멀티스레딩 (threading 모듈)

`threading` 모듈을 사용하여 스레드를 생성하고,
Lock으로 공유 자원을 안전하게 보호하는 방법을 실습합니다.

### 2.1 Thread 생성 · start() · join()

스레드를 만들고 실행하는 기본 흐름:
1. `threading.Thread(target=함수, args=인수)` 로 Thread 객체 생성
2. `.start()` 로 스레드 시작
3. `.join()` 으로 스레드가 끝날 때까지 메인 프로그램 대기

In [ ]:
import threading
import time

def sensor_reader(name):
    print(f"{name} 읽기 시작")
    time.sleep(2)  # 센서 응답 대기 시뮬레이션
    print(f"{name} 읽기 완료")

# 1. Thread 생성 (target=실행할 함수, args=인수)
t = threading.Thread(target=sensor_reader, args=("Lidar",))

# 2. 스레드 시작
t.start()

# 3. 스레드가 끝날 때까지 메인 프로그램 대기
t.join()
print("모든 센서 작업 완료")

### 2.2 Daemon(데몬) 스레드

**메인 프로그램 종료 시 함께 강제 종료되는 스레드**

- **용도**
    - 로봇의 상태를 주기적으로 로깅
    - 백그라운드에서 하트비트(Heartbeat) 신호 전송
    - 메인 프로그램이 꺼졌는데 로깅 스레드가 살아있으면 프로그램이 종료 안 됨

- `daemon=True` 로 생성 시 설정
- 메인 프로그램 종료 시 데몬 스레드도 자동 종료됨

In [ ]:
# 03.02.01.Python-Thread-Daemon.py

# 메인 프로그램이 0.35초 후 종료되면 데몬 스레드도 함께 종료됨
time.sleep(0.35)
print("메인 프로그램 종료 → 데몬 스레드 자동 종료")

### 2.3 Lock을 이용한 공유 자원 보호

**Race Condition (경쟁 상태)**: 여러 스레드가 하나의 변수·하드웨어 포트에 동시 접근 시 데이터가 꼬이는 문제

**해결책: Lock(자물쇠) 사용**
- `with lock:` 블록 안에서만 공유 자원 접근
- 한 스레드가 작업 중이면 다른 스레드는 대기

In [ ]:
import threading

lock = threading.Lock()
telemetry_data = 0

def update_data():
    global telemetry_data
    with lock:  # 자물쇠 잠금 (작업 후 자동으로 풀림)
        # 안전하게 공유 자원 수정
        telemetry_data += 1

# 여러 스레드가 동시에 telemetry_data를 안전하게 수정하는 예시
threads = [threading.Thread(target=update_data) for _ in range(5)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print(f"최종 telemetry_data: {telemetry_data}")  # 5

### 2.4 스레드 안전(Thread-safe) 코드 작성 방법

**3가지 원칙**

1. **Lock 사용**
    - 공유 변수에 접근할 때 반드시 `threading.Lock()` 사용
    - 한 번에 하나의 스레드만 접근하도록 제어

2. **Queue 활용**
    - `queue.Queue` 모듈: 내부적으로 Lock이 구현되어 있음
    - 스레드 간에 안전하게 데이터를 주고받을 수 있음

3. **공유 상태 최소화**
    - 가급적 스레드 간에 데이터를 공유하지 않도록 설계
    - 독립적인 결과물만 내놓는 구조가 가장 안전

### 2.5 입문자가 자주 하는 실수 (멀티스레딩)

- **경쟁 상태 (Race Condition)**
    - 두 개 이상의 스레드가 하나의 데이터를 동시에 수정
    - 실행 타이밍에 따라 결과가 매번 달라지는 현상
    - **반드시 Lock을 사용해야 함**

- **데드락 (Deadlock)**
    - 두 스레드가 서로 상대방이 가진 Lock이 풀리기만을 기다리며 무한히 멈춤
    - 예시
        - A스레드: 자물쇠1 보유, 자물쇠2 대기
        - B스레드: 자물쇠2 보유, 자물쇠1 대기
        - → 서로 무한 대기 상태 발생

### 2.6 (종합 예제) 여러 센서를 동시에 읽기

Lock으로 보호되는 공유 딕셔너리에 Lidar·Ultrasonic 스레드가 동시에 데이터를 갱신하고,
메인 루프에서 주기적으로 읽어 출력합니다.

> **실습 팁:** Lock을 제거한 후 실행해 보고 어떤 문제가 생기는지 직접 확인해 보세요

In [ ]:
# 03.02.02.Python-Multi-Thread.py

---
## Part 3: 멀티프로세싱 (multiprocessing 모듈)

`multiprocessing` 모듈을 사용하여 독립된 프로세스를 생성하고,
Queue·Event로 프로세스 간 통신하는 방법을 실습합니다.

### 3.1 multiprocessing 기본 사용법

- **Process 생성 · start() · join()**
    - `Process` 객체 생성 시 실행할 함수(`target`)와 인자(`args`)를 지정
    - `start()`: 프로세스 시작
    - `join()`: 해당 프로세스가 종료될 때까지 부모 프로세스 대기

- **Queue를 이용한 프로세스 간 통신**
    - 프로세스는 메모리를 공유하지 않으므로 `Queue`("우체통")로 데이터 교환
    - `put()`: 데이터 전송 / `get()`: 데이터 수신

- **Event를 이용한 프로세스 간 신호**
    - 모든 프로세스가 공유하는 "정지 버튼"
    - `set()`: 신호 발송 / `is_set()`: 신호 확인
    - 비상 정지, 일괄 종료에 활용

### 3.2 로봇 개발에서의 멀티프로세싱 패턴

**"3-프로세스 자동화 패턴"**

- **센서/카메라 프로세스**
    - 최대한 빠른 속도(예: 60 FPS)로 데이터를 읽어 Queue에 전송

- **추론/분석 프로세스**
    - AI 모델(CPU/GPU 집약적)로 데이터 분석
    - 분석 속도가 느려도(예: 10 FPS) 다른 프로세스를 방해하지 않음

- **제어 프로세스**
    - 로봇의 물리적 움직임 관리
    - AI 추론이 늦어지더라도 독립적으로 동작
    - 일정 주기(Heartbeat)로 하드웨어와 통신하여 시스템 안정성 유지

### 3.3 Pool을 이용한 병렬 처리

**동일한 작업을 여러 입력값에 대해 병렬 실행 (데이터 병렬 처리)**

- `Pool(processes=N)`: N개의 워커 프로세스 풀 생성
- `map(함수, 데이터_리스트)`: 데이터를 분배하여 처리 후 결과 리스트 반환

> 대량의 센서 데이터 필터링, 복잡한 수치 계산에 효율적

In [ ]:
# 03.02.03.Python-Process-Pool.py

### 3.4 입문자가 자주 하는 실수 (멀티프로세싱)

- **`if __name__ == '__main__':` 누락**
    - 윈도우·macOS에서 이 조건문 없이 프로세스 생성 시
    - → 프로세스가 무한히 복제되는 런타임 오류 발생

- **변수 직접 공유 시도**
    - 프로세스는 독립된 메모리를 가짐
    - 전역 변수를 수정해도 다른 프로세스에는 반영되지 않음
    - → 반드시 `Queue`나 `Pipe` 사용

- **무분별한 `terminate()` 사용**
    - `terminate()`로 프로세스를 강제 종료하면
    - → 해당 프로세스가 보유한 Lock·Queue가 깨져 전체 시스템 **데드락** 위험

### 3.5 (종합 예제) AI 추론과 센서 읽기 분리

센서 프로세스가 100Hz로 데이터를 Queue에 넣고,
AI 추론 프로세스가 Queue에서 꺼내 분석합니다.
`multiprocessing.Event`로 두 프로세스를 동시에 안전하게 종료합니다.

In [ ]:
# 03.02.04.Python-Process-Queue.py

---
## Part 4: 퀴즈 · 복습 문제

멀티스레딩과 멀티프로세싱의 핵심 개념을 코드로 확인합니다.

### 4.1 코드 읽기 문제


In [ ]:
# **문제 1**: 다음 코드의 출력 결과는?
# ```python
# import threading
# def print_hello():
#     print("Hello")
# t = threading.Thread(target=print_hello)
# t.start()
# t.join()
# print("Done")
# ```

In [ ]:
# **문제 2**: 다음 멀티프로세싱 코드에서 반드시 수정해야 할 부분은?
# ```python
# from multiprocessing import Process
# def task():
#     print("Working...")
# p = Process(target=task)
# p.start()
# ```

In [ ]:
# **문제 3**: 다음 코드에서 프로세스 간 데이터 공유가 안 되는 이유는?
# ```python
# from multiprocessing import Process
# counter = 0
# def increment():
#     global counter
#     counter += 1
# p = Process(target=increment)
# p.start()
# p.join()
# print(counter)  # 출력: 0
# ```

In [ ]:
# **문제 4**: 다음 코드에서 `Queue`의 역할은?
# ```python
# from multiprocessing import Process, Queue
# def worker(q):
#     q.put("Result from worker")
# queue = Queue()
# p = Process(target=worker, args=(queue,))
# p.start()
# print(queue.get())
# ```

---
## 도전 실습: 멀티스레딩 + Queue를 활용한 센서 파이프라인

아래 요구사항을 만족하는 멀티스레딩 기반 센서 파이프라인을 구현하세요.

**요구사항**
- 생산자(producer) 스레드: 0.1초마다 랜덤 센서 값을 생성하여 `queue.Queue`에 넣기
- 소비자(consumer) 스레드: Queue에서 값을 꺼내 처리 결과 출력
- 메인 스레드: 3초 후 두 스레드를 안전하게 종료

> **힌트**: `queue.Queue`는 내부적으로 Lock이 구현되어 있어 별도 Lock 없이 Thread-safe하게 사용 가능합니다.

In [ ]:
# 03.02.05.PYthon-Thread-Practice.py